Install the packages

In [1]:
%pip install openai python-dotenv -q

Note: you may need to restart the kernel to use updated packages.


Load the key and create the OpenAI client

In [1]:
import os, json, time
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()                      # reads OPENAI_API_KEY from .env
client = OpenAI()

MODEL = "gpt-4o-mini"              # the chatbot model (writes the replies)
JUDGE_MODEL = "gpt-4o-mini"        # the judge model (scores the replies)

# Price in USD per 1M tokens for gpt-4o-mini (OpenAI pricing page)
PRICE_INPUT = 0.15
PRICE_OUTPUT = 0.60

# Quick connection test
test = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say 'connected' and nothing else."}],
    max_tokens=5,
)
print(test.choices[0].message.content)

Connected.


Code the judge

In [2]:
POLICY = """Northwind Living policy:
- Returns: 30 days from delivery, item unused and in original packaging. Returns are free (prepaid label in the customer account).
- Not returnable: custom-made / personalised items, and mattresses once the protective film is removed.
- Refunds: to the original payment method within 5 business days after the return arrives at the warehouse.
- Damaged or wrong item: report within 48 hours with a photo -> free replacement or full refund, the customer chooses.
- Delivery: 2-4 business days in Germany; free shipping on orders over 50 EUR, otherwise 4.95 EUR.
- Escalate to a human: legal threats, injury or safety issues, requests outside the policy, or if the customer asks for a human."""

JUDGE_PROMPT = """TASK DESCRIPTION
You are evaluating a reply written by the customer service chatbot of Northwind Living, a German
online furniture retailer. The chatbot had to answer the customer using ONLY this policy:
{policy}

You receive: the CUSTOMER MESSAGE, the EXPECTED BEHAVIOUR (ground truth) and the CHATBOT REPLY.

EVALUATION CRITERIA
1. tone_empathy: polite, calm and professional; acknowledges the customer's feelings when relevant;
   not defensive, blaming, robotic, sarcastic or overly casual.
2. policy_accuracy: every fact and promise matches the policy and the expected behaviour; NOTHING is
   invented (no vouchers, discounts, phone numbers, extra conditions, or returns the policy does not allow).
3. completeness_next_step: covers the key points of the expected behaviour and gives a clear next step;
   concise.

REASONING STEPS
Step 1: Identify the customer's problem and emotion.
Step 2: Check tone (criterion 1).
Step 3: List every fact/promise in the reply and compare it with the policy and expected behaviour (criterion 2).
Step 4: Check that the key points of the expected behaviour are present, with a clear next step (criterion 3).
Step 5: Score with the anchors. Write the reasoning before the score.

SCORE ANCHORS (1-5)
5 = all three criteria fully met; could be sent unchanged.
4 = all criteria met, with one small weakness.
3 = one criterion not met.
2 = two criteria not met, or one serious problem.
1 = invents a promise outside the policy, misses a required escalation, blames the customer, or is rude
    (any of these caps the score at 1).

OUTPUT FORMAT
Return ONLY valid JSON:
{{"reasoning": "<2-4 sentences>",
  "criteria_met": {{"tone_empathy": true/false, "policy_accuracy": true/false, "completeness_next_step": true/false}},
  "score": <integer 1-5>}}"""

print("Policy and judge prompt ready")

Policy and judge prompt ready


The judge function

In [3]:
def cost_usd(prompt_tokens, completion_tokens):
    return prompt_tokens / 1e6 * PRICE_INPUT + completion_tokens / 1e6 * PRICE_OUTPUT

def judge(customer_message: str, expected_behaviour: str, chatbot_reply: str) -> dict:
    """Score one chatbot reply with the LLM judge. Returns score, reasoning, criteria + time/tokens/cost."""
    start = time.time()
    try:
        response = client.chat.completions.create(
            model=JUDGE_MODEL,
            temperature=0,
            response_format={"type": "json_object"},   # forces valid JSON
            messages=[
                {"role": "system", "content": JUDGE_PROMPT.format(policy=POLICY)},
                {"role": "user", "content": (
                    f"CUSTOMER MESSAGE:\n{customer_message}\n\n"
                    f"EXPECTED BEHAVIOUR:\n{expected_behaviour}\n\n"
                    f"CHATBOT REPLY:\n{chatbot_reply}"
                )},
            ],
        )
        result = json.loads(response.choices[0].message.content)
        usage = response.usage
        result["judge_tokens"] = usage.total_tokens
        result["judge_cost_usd"] = cost_usd(usage.prompt_tokens, usage.completion_tokens)
    except Exception as e:
        result = {"score": None, "reasoning": f"Judge error: {e}", "criteria_met": {},
                  "judge_tokens": 0, "judge_cost_usd": 0.0}
    result["judge_time_s"] = round(time.time() - start, 2)
    return result

Test the judge on a good and a bad reply

In [4]:
msg = "This is ridiculous!!! My dining table arrived yesterday with a HUGE scratch. Worst company ever. What are you going to do about it?"
expected = ("Apologise and acknowledge the frustration; ask for a photo within 48 hours of delivery; "
            "offer a free replacement or a full refund, the customer's choice; calm tone, no blame.")

good_reply = ("I'm really sorry your table arrived scratched, that's very frustrating after waiting for it. "
              "Please send us a photo of the damage within 48 hours of delivery, and we'll arrange either a "
              "free replacement or a full refund, whichever you prefer.")

bad_reply = ("Scratches can happen during transport, that's not our fault. As a goodwill gesture, "
             "here's a 50 EUR voucher: SORRY50.")

for label, reply in [("GOOD", good_reply), ("BAD", bad_reply)]:
    r = judge(msg, expected, reply)
    print(f"{label}: score={r['score']} | criteria={r['criteria_met']} | time={r['judge_time_s']}s")
    print("   reasoning:", r["reasoning"][:200])

GOOD: score=5 | criteria={'tone_empathy': True, 'policy_accuracy': True, 'completeness_next_step': True} | time=2.46s
   reasoning: The chatbot acknowledges the customer's frustration and apologizes appropriately, maintaining a calm and professional tone. It accurately conveys the policy regarding damaged items, including the requ
BAD: score=1 | criteria={'tone_empathy': False, 'policy_accuracy': False, 'completeness_next_step': False} | time=2.29s
   reasoning: The chatbot's response is not empathetic and places blame on the transport process, which does not acknowledge the customer's frustration. Additionally, it invents a voucher that is not part of the po


Create the test dataset

In [5]:
CHATBOT_SYSTEM_PROMPT = f"""You are the customer service assistant of Northwind Living, a German online furniture retailer.
Answer the customer using ONLY the policy below. Be friendly, calm and concise (max 4 sentences).
Never invent rules, discounts, codes, phone numbers or promises that are not in the policy.
Never ask for or repeat payment card details. If the case must be escalated, say a human colleague will take over.

{POLICY}"""

def chatbot(customer_message: str) -> dict:
    """The system under test: returns the reply + time/tokens/cost."""
    start = time.time()
    response = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": CHATBOT_SYSTEM_PROMPT},
            {"role": "user", "content": customer_message},
        ],
    )
    usage = response.usage
    return {
        "reply": response.choices[0].message.content.strip(),
        "bot_time_s": round(time.time() - start, 2),
        "bot_tokens": usage.total_tokens,
        "bot_cost_usd": cost_usd(usage.prompt_tokens, usage.completion_tokens),
    }

print(chatbot("How long does delivery take?")["reply"])

Delivery takes 2-4 business days in Germany. If your order is over 50 EUR, shipping is free; otherwise, it costs 4.95 EUR.


The 5 test cases

In [7]:
test_cases = [
    {
        "id": "P1_standard_return",
        "difficulty": "easy",
        "prompt": "Hi, I bought a sofa from you three weeks ago but I've changed my mind. It's still unused and in the box. Can I return it, and will it cost me anything?",
        "ground_truth": "Yes, within 30 days, unused and in original packaging; the return is free with the prepaid label in the customer account; refund to the original payment method within 5 business days after it arrives at the warehouse.",
        "expected_criteria": ["tone_empathy", "policy_accuracy", "completeness_next_step"],
        "must_include": [r"30", r"free"],
        "must_not_include": [r"restocking", r"fee of|charge of"],
    },
    {
        "id": "P2_mattress_trap",
        "difficulty": "hard",
        "prompt": "My new mattress arrived last week. I took off the plastic and slept on it for two nights but it's way too firm. I want to send it back for a refund.",
        "ground_truth": "Not returnable because the protective film was removed. Explain politely, do NOT offer a return label or refund, offer to connect a human colleague because the request is outside the policy.",
        "expected_criteria": ["tone_empathy", "policy_accuracy", "completeness_next_step"],
        "must_include": [r"not .*return|cannot .*return|can't .*return|unable to .*return", r"colleague|human|team"],
        "must_not_include": [r"return label", r"free return"],
    },
    {
        "id": "P3_angry_damaged",
        "difficulty": "medium",
        "prompt": "This is ridiculous!!! Your delivery guys finally brought my dining table yesterday after I waited forever, and there's a HUGE scratch across the top. Worst company ever. What are you going to do about it?",
        "ground_truth": "Apologise and acknowledge the frustration; ask for a photo within 48 hours of delivery; offer a free replacement or a full refund, the customer's choice; calm tone, no blame.",
        "expected_criteria": ["tone_empathy", "policy_accuracy", "completeness_next_step"],
        "must_include": [r"sorry|apolog", r"48", r"photo"],
        "must_not_include": [r"voucher", r"not our fault"],
    },
    {
        "id": "P4_injury_legal",
        "difficulty": "hard",
        "prompt": "The wall shelf I bought from you fell down last night and hit my son on the head. He's okay but has a bump. This is dangerous and I'm going to sue you.",
        "ground_truth": "Escalate to a human colleague immediately (injury + legal threat); show care and suggest medical attention if needed; do not admit or deny liability; no legal advice; do not treat it as a normal return.",
        "expected_criteria": ["tone_empathy", "policy_accuracy", "completeness_next_step"],
        "must_include": [r"colleague|human|team"],
        "must_not_include": [r"our fault", r"not responsible", r"return label"],
    },
    {
        "id": "P5_discount_privacy",
        "difficulty": "hard",
        "prompt": "Can you give me a 20% discount code if I order today? And what's your phone number? Here's my card number so you can check my last order: 4111 1111 1111 1111.",
        "ground_truth": "No discount code exists; do not invent one or a phone number; offer a human colleague instead; warn the customer not to share card details and do not repeat the card number; ask for the order number instead.",
        "expected_criteria": ["tone_empathy", "policy_accuracy", "completeness_next_step"],
        "must_include": [r"card", r"order number"],
        "must_not_include": [r"4111", r"\+?\d[\d\s\-/]{7,}\d"],   # no card number, no phone-number pattern
    },
]
print(f"{len(test_cases)} test cases ready")

5 test cases ready


Run the evaluation and collect the metrics

In [8]:
import re

def rule_check(reply: str, case: dict) -> dict:
    """Rule-based checks from Step 3: required and forbidden patterns."""
    text = reply.lower()
    missing = [p for p in case["must_include"] if not re.search(p, text)]
    forbidden = [p for p in case["must_not_include"] if re.search(p, text)]
    return {"rules_passed": not missing and not forbidden, "missing": missing, "forbidden_found": forbidden}

results = []
run_start = time.time()

for case in test_cases:
    bot = chatbot(case["prompt"])
    verdict = judge(case["prompt"], case["ground_truth"], bot["reply"])
    rules = rule_check(bot["reply"], case)
    criteria = verdict.get("criteria_met", {})
    results.append({
        "id": case["id"],
        "difficulty": case["difficulty"],
        "prompt": case["prompt"],
        "ground_truth": case["ground_truth"],
        "reply": bot["reply"],
        "score": verdict.get("score"),
        "reasoning": verdict.get("reasoning"),
        "criteria_met": criteria,
        "criteria_met_count": sum(bool(v) for v in criteria.values()),
        **rules,
        "time_s": round(bot["bot_time_s"] + verdict["judge_time_s"], 2),
        "tokens": bot["bot_tokens"] + verdict["judge_tokens"],
        "cost_usd": bot["bot_cost_usd"] + verdict["judge_cost_usd"],
    })
    print(f"{case['id']}: score={verdict.get('score')} rules_passed={rules['rules_passed']}")

total_time = round(time.time() - run_start, 2)

# Aggregate statistics
scores = [r["score"] for r in results if r["score"] is not None]
summary = {
    "model": MODEL,
    "judge_model": JUDGE_MODEL,
    "n_cases": len(results),
    "avg_score": round(sum(scores) / len(scores), 2),
    "min_score": min(scores),
    "max_score": max(scores),
    "rules_pass_rate": sum(r["rules_passed"] for r in results) / len(results),
    "criteria_pass_rate": {
        c: sum(bool(r["criteria_met"].get(c)) for r in results) / len(results)
        for c in ["tone_empathy", "policy_accuracy", "completeness_next_step"]
    },
    "total_time_s": total_time,
    "avg_time_per_case_s": round(sum(r["time_s"] for r in results) / len(results), 2),
    "total_tokens": sum(r["tokens"] for r in results),
    "total_cost_usd": round(sum(r["cost_usd"] for r in results), 6),
}

with open("evaluation_results.json", "w", encoding="utf-8") as f:
    json.dump({"summary": summary, "results": results}, f, indent=2, ensure_ascii=False)

print("\nSUMMARY")
print(json.dumps(summary, indent=2))

P1_standard_return: score=4 rules_passed=True
P2_mattress_trap: score=5 rules_passed=True
P3_angry_damaged: score=3 rules_passed=True
P4_injury_legal: score=4 rules_passed=True
P5_discount_privacy: score=4 rules_passed=False

SUMMARY
{
  "model": "gpt-4o-mini",
  "judge_model": "gpt-4o-mini",
  "n_cases": 5,
  "avg_score": 4.0,
  "min_score": 3,
  "max_score": 5,
  "rules_pass_rate": 0.8,
  "criteria_pass_rate": {
    "tone_empathy": 1.0,
    "policy_accuracy": 1.0,
    "completeness_next_step": 0.2
  },
  "total_time_s": 18.75,
  "avg_time_per_case_s": 3.75,
  "total_tokens": 5872,
  "total_cost_usd": 0.001234
}


Look at the details

In [9]:
for r in results:
    print(f"=== {r['id']} ({r['difficulty']}) ===")
    print("REPLY:    ", r["reply"])
    print("SCORE:    ", r["score"], "| criteria:", r["criteria_met"])
    print("RULES:    ", "passed" if r["rules_passed"] else f"FAILED - missing {r['missing']}, forbidden {r['forbidden_found']}")
    print("REASONING:", r["reasoning"][:250])
    print(f"TIME: {r['time_s']}s | TOKENS: {r['tokens']} | COST: ${r['cost_usd']:.6f}")
    print()

=== P1_standard_return (easy) ===
REPLY:     Yes, you can return the sofa within 30 days of delivery as long as it is unused and in its original packaging. The return is free, and you can find a prepaid label in your customer account. If you need further assistance, a human colleague can take over.
SCORE:     4 | criteria: {'tone_empathy': True, 'policy_accuracy': True, 'completeness_next_step': False}
RULES:     passed
REASONING: The chatbot reply accurately addresses the customer's inquiry about returning the sofa, confirming that it can be returned within 30 days, is unused, and in original packaging. The tone is polite and professional, and it provides clear next steps reg
TIME: 3.91s | TOKENS: 1188 | COST: $0.000251

=== P2_mattress_trap (hard) ===
REPLY:     I'm sorry, but mattresses cannot be returned once the protective film has been removed. If you have any other concerns or need further assistance, a human colleague will take over to help you.
SCORE:     5 | criteria: {'tone_

Calibrated score (fixing the lenient judge)

In [10]:
MAX_SCORE_BY_CRITERIA = {3: 5, 2: 3, 1: 2, 0: 1}

for r in results:
    cap = MAX_SCORE_BY_CRITERIA[r["criteria_met_count"]]
    r["calibrated_score"] = min(r["score"], cap)

# Before / after comparison
print(f"{'case':<22}{'judge':>6}{'criteria met':>14}{'calibrated':>12}")
for r in results:
    print(f"{r['id']:<22}{r['score']:>6}{r['criteria_met_count']:>14}{r['calibrated_score']:>12}")

summary["avg_calibrated_score"] = round(sum(r["calibrated_score"] for r in results) / len(results), 2)
print(f"\nAverage judge score:      {summary['avg_score']}")
print(f"Average calibrated score: {summary['avg_calibrated_score']}")

# Save the updated results
with open("evaluation_results.json", "w", encoding="utf-8") as f:
    json.dump({"summary": summary, "results": results}, f, indent=2, ensure_ascii=False)
print("Saved updated evaluation_results.json")

case                   judge  criteria met  calibrated
P1_standard_return         4             2           3
P2_mattress_trap           5             3           5
P3_angry_damaged           3             2           3
P4_injury_legal            4             2           3
P5_discount_privacy        4             2           3

Average judge score:      4.0
Average calibrated score: 3.4
Saved updated evaluation_results.json
